# Peterson-McCabe: Sentence Contrast in Free Personal Narratives

**Question:** Does distant context help predict the next sentence more in older children's free narratives?

**Why v5 (contrast) instead of v6 (influence)?** The leave-one-out influence method is confounded by text length — shorter texts inflate influence values. The contrast method measures in tokens and avoids this issue.

**Data:** Peterson & McCabe personal narratives, ages 4-9. Free speech with no external scaffolding.

**Impostors:** From other children in the same age bin — controls for age-typical language while testing document-specific coherence.

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path
import json, math, time, gc, os, re, torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("Imports OK")

In [ ]:
IN_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DATA = Path("/content/drive/MyDrive/LRTIA/Data/petmcc_processed")
    DRIVE_RESULTS = Path("/content/drive/MyDrive/LRTIA/Results/PetMcc_contrast")
    if (DRIVE_DATA / "transcripts.jsonl").exists():
        DATA_DIR = DRIVE_DATA
    else:
        LOCAL_DATA = Path("/content/data/petmcc_processed")
        if not (LOCAL_DATA / "transcripts.jsonl").exists():
            LOCAL_DATA.mkdir(parents=True, exist_ok=True)
            from google.colab import files
            print("Upload transcripts.jsonl:")
            uploaded = files.upload()
            for fname in uploaded:
                with open(LOCAL_DATA / fname, 'wb') as f:
                    f.write(uploaded[fname])
        DATA_DIR = LOCAL_DATA
    BASE_DIR = DRIVE_RESULTS
    BASE_DIR.mkdir(parents=True, exist_ok=True)
else:
    BASE_DIR = Path("../results/petmcc_contrast")
    DATA_DIR = Path("../data/petmcc_processed")
    BASE_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_DIR: {DATA_DIR}")
print(f"BASE_DIR: {BASE_DIR}")

MODEL_NAME = "mistralai/Mistral-7B-v0.1"
USE_4BIT = True

WINDOWS = [4, 8, 16, 32, 64]  # shorter texts so smaller max window
N_IMPOSTORS = 3
MAX_TARGET_SENTS = 3
MIN_WORDS = 50
MIN_SENTS = 4
RANDOM_SEED = 42

AGE_BINS = [
    (48, 66, '4-5y'),
    (67, 84, '5-7y'),
    (85, 113, '7-9y'),
]

DOMAINS = ['personal_narrative']  # single domain for impostor pooling

print(f"Windows: {WINDOWS}")
print(f"Age bins: {[b[2] for b in AGE_BINS]}")

In [ ]:
corpus_all = []
with open(DATA_DIR / "transcripts.jsonl") as f:
    for line in f:
        d = json.loads(line)
        pop = json.loads(d['population'])
        d['age_months'] = pop.get('age_months', None)
        d['sex'] = pop.get('sex', None)
        d['word_count'] = len(d['text'].split())
        d['domain'] = 'personal_narrative'
        d['population'] = 'human'
        d['model'] = 'human'
        sents = re.split(r'(?<=[.!?])\s+', d['text'].strip())
        d['n_sents'] = len([s for s in sents if len(s.strip().split()) >= 3])
        corpus_all.append(d)

corpus_all = [d for d in corpus_all
              if d['age_months'] is not None
              and d['word_count'] >= MIN_WORDS
              and d['n_sents'] >= MIN_SENTS]

for d in corpus_all:
    d['age_bin'] = None
    for lo, hi, label in AGE_BINS:
        if lo <= d['age_months'] <= hi:
            d['age_bin'] = label
            break
corpus = [d for d in corpus_all if d['age_bin'] is not None]

print(f"Total: {len(corpus)} transcripts")

# Build impostor pools by age bin
impostor_pools = {}
for lo, hi, label in AGE_BINS:
    pool = {}
    for d in corpus:
        if d['age_bin'] == label:
            sents = re.split(r'(?<=[.!?])\s+', d['text'].strip())
            sents = [s.strip() for s in sents if len(s.strip().split()) >= 3]
            if len(sents) >= 3:
                pool[d['doc_id']] = sents
    impostor_pools[label] = pool
    n_sents = sum(len(v) for v in pool.values())
    print(f"  {label}: {len(pool)} docs, {n_sents} sentences in pool")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto")
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
model.eval()
print("Model loaded")

In [ ]:
@torch.no_grad()
def compute_ppl_on_tokens(context_ids, target_ids):
    full_ids = context_ids + target_ids
    input_ids = torch.tensor([full_ids], device=model.device)
    outputs = model(input_ids)
    logits = outputs.logits[0]
    target_start = len(context_ids)
    total_loss = 0.0
    count = 0
    for i in range(target_start, len(full_ids) - 1):
        log_probs = torch.log_softmax(logits[i], dim=-1)
        total_loss += -log_probs[full_ids[i + 1]].item()
        count += 1
    del outputs, logits
    torch.cuda.empty_cache()
    return math.exp(total_loss / count) if count > 0 else float('inf')


def split_into_sentences(text, tokenizer):
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    result = []
    current_pos = 0
    for sent_text in sentences:
        sent_text = sent_text.strip()
        if len(sent_text.split()) < 3:
            sent_ids = tokenizer.encode(sent_text, add_special_tokens=False)
            current_pos += len(sent_ids)
            continue
        sent_ids = tokenizer.encode(sent_text, add_special_tokens=False)
        if len(sent_ids) >= 2:
            result.append({
                'text': sent_text,
                'ids': sent_ids,
                'start': current_pos,
                'end': current_pos + len(sent_ids),
            })
        current_pos += len(sent_ids)
    return result


def get_impostor_sentences(age_bin, exclude_doc_id, rng, n=3):
    pool = impostor_pools.get(age_bin, {})
    candidates = []
    for doc_id, sents in pool.items():
        if doc_id != exclude_doc_id:
            candidates.extend(sents)
    if len(candidates) < n:
        return []
    return list(rng.choice(candidates, size=n, replace=False))


def compute_contrast_curve(doc, rng):
    text = doc['text']
    age_bin = doc['age_bin']
    doc_id = doc['doc_id']
    full_ids = tokenizer.encode(text, add_special_tokens=False)
    n_tokens = len(full_ids)

    sentences = split_into_sentences(text, tokenizer)
    if len(sentences) < 4:
        return None

    mid = len(sentences) // 2
    target_sents = sentences[mid:]
    if len(target_sents) < 2:
        return None
    if len(target_sents) > MAX_TARGET_SENTS:
        idx = rng.choice(len(target_sents), MAX_TARGET_SENTS, replace=False)
        target_sents = [target_sents[j] for j in idx]

    results_by_W = {}
    for W in WINDOWS:
        contrast_scores = []
        for tgt in target_sents:
            sent_start = tgt['start']
            sent_end = tgt['end']
            if sent_end > n_tokens or sent_start < W:
                continue
            context_start = max(0, sent_start - W)
            context_ids = full_ids[context_start:sent_start]
            real_sent_ids = tgt['ids']
            if len(real_sent_ids) < 2 or len(context_ids) < 2:
                continue

            ppl_real = compute_ppl_on_tokens(context_ids, real_sent_ids)
            if math.isinf(ppl_real) or ppl_real <= 0:
                continue

            imp_texts = get_impostor_sentences(age_bin, doc_id, rng, n=N_IMPOSTORS)
            if len(imp_texts) < 2:
                continue

            ppl_impostors = []
            for imp_text in imp_texts:
                imp_ids = tokenizer.encode(imp_text, add_special_tokens=False)
                if len(imp_ids) < 2:
                    continue
                ppl_imp = compute_ppl_on_tokens(context_ids, imp_ids)
                if not math.isinf(ppl_imp) and ppl_imp > 0:
                    ppl_impostors.append(ppl_imp)

            if len(ppl_impostors) < 2:
                continue

            contrast = math.log(np.mean(ppl_impostors) / ppl_real)
            contrast_scores.append(contrast)

        if len(contrast_scores) >= 1:
            results_by_W[W] = np.mean(contrast_scores)

    return results_by_W if len(results_by_W) >= 3 else None


print("Functions defined")

In [ ]:
results_path = BASE_DIR / "petmcc_contrast_results.csv"

if results_path.exists():
    df = pd.read_csv(results_path)
    print(f"Loaded existing results: {len(df)} rows")
else:
    rng = np.random.RandomState(RANDOM_SEED)
    results = []
    skipped = 0
    for doc in tqdm(corpus, desc="Transcripts"):
        curve = compute_contrast_curve(doc, rng)
        if curve is None:
            skipped += 1
            continue
        row = {
            'doc_id': doc['doc_id'],
            'age_months': doc['age_months'],
            'age_bin': doc['age_bin'],
            'word_count': doc['word_count'],
        }
        for W, val in curve.items():
            row[f'contrast_W{W}'] = val
        results.append(row)

    df = pd.DataFrame(results)
    df.to_csv(results_path, index=False)
    print(f"Processed {len(df)} transcripts ({skipped} skipped)")
    print(f"Saved to {results_path}")

print(f"By age bin:")
for _, _, label in AGE_BINS:
    print(f"  {label}: {len(df[df.age_bin == label])}")

In [ ]:
contrast_cols = [f'contrast_W{w}' for w in WINDOWS]
age_colors = {'4-5y': '#e74c3c', '5-7y': '#f39c12', '7-9y': '#27ae60'}

fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# A: Raw contrast by age
ax = axes[0, 0]
for _, _, label in AGE_BINS:
    sub = df[df.age_bin == label]
    vals = [sub[c].mean() for c in contrast_cols if c in sub.columns]
    sems = [sub[c].sem() for c in contrast_cols if c in sub.columns]
    ws = [w for w in WINDOWS if f'contrast_W{w}' in sub.columns]
    ax.errorbar(ws, vals, yerr=sems, fmt='o-', color=age_colors[label],
                linewidth=2, markersize=5, capsize=3, label=f'{label} (n={len(sub)})')
    ax.fill_between(ws, np.array(vals)-np.array(sems), np.array(vals)+np.array(sems),
                    color=age_colors[label], alpha=0.15)
ax.set_xscale('log', base=2)
ax.set_xlabel('Context Window (tokens)')
ax.set_ylabel('Contrast Score')
ax.set_title('A. Sentence Contrast by Age (Free Narratives)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.2)

# B: Normalized curves
ax = axes[0, 1]
for _, _, label in AGE_BINS:
    sub = df[df.age_bin == label]
    vals = np.array([sub[c].mean() for c in contrast_cols if c in sub.columns])
    if len(vals) >= 3 and vals[-1] - vals[0] > 0.01:
        norm = (vals - vals[0]) / (vals[-1] - vals[0])
        ws = [w for w in WINDOWS if f'contrast_W{w}' in sub.columns]
        ax.plot(ws, norm, 'o-', color=age_colors[label], linewidth=2, markersize=5, label=label)
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.4)
ax.plot([WINDOWS[0], WINDOWS[-1]], [0, 1], 'k--', alpha=0.2)
ax.set_xscale('log', base=2)
ax.set_ylim(-0.1, 1.1)
ax.set_xlabel('Context Window (tokens)')
ax.set_ylabel('Fraction of Total Contrast Gain')
ax.set_title('B. Normalized Curves by Age', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.2)

# C: Age continuous vs contrast at each window
ax = axes[1, 0]
for w in WINDOWS:
    col = f'contrast_W{w}'
    if col in df.columns:
        valid = df[[col, 'age_months']].dropna()
        if len(valid) >= 10:
            r, p = stats.pearsonr(valid['age_months'], valid[col])
            ax.scatter(valid['age_months'], valid[col], alpha=0.2, s=10)
            slope, intercept, _, _, _ = stats.linregress(valid['age_months'], valid[col])
            x_line = np.array([valid['age_months'].min(), valid['age_months'].max()])
            ax.plot(x_line, intercept + slope * x_line, '--', linewidth=2,
                    label=f'W{w}: r={r:.3f}, p={p:.3f}')
ax.set_xlabel('Age (months)')
ax.set_ylabel('Contrast Score')
ax.set_title('C. Contrast vs Age (continuous)', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.2)

# D: Delta (max_W - W4) vs age
ax = axes[1, 1]
max_w = max(WINDOWS)
col_min = f'contrast_W{WINDOWS[0]}'
col_max = f'contrast_W{max_w}'
if col_min in df.columns and col_max in df.columns:
    df['delta'] = df[col_max] - df[col_min]
    valid = df[['delta', 'age_months']].dropna()
    for _, _, label in AGE_BINS:
        sub = df[df.age_bin == label]
        ax.scatter(sub['age_months'], sub['delta'], alpha=0.4, s=20, color=age_colors[label], label=label)
    r, p = stats.pearsonr(valid['age_months'], valid['delta'])
    slope, intercept, _, _, _ = stats.linregress(valid['age_months'], valid['delta'])
    x_line = np.array([valid['age_months'].min(), valid['age_months'].max()])
    ax.plot(x_line, intercept + slope * x_line, 'k--', linewidth=2)
    ax.set_title(f'D. Context Benefit (W{max_w}-W{WINDOWS[0]}) vs Age (r={r:.3f}, p={p:.4f})', fontweight='bold')
ax.set_xlabel('Age (months)')
ax.set_ylabel(f'Contrast gain (W{max_w} - W{WINDOWS[0]})')
ax.legend()
ax.grid(True, alpha=0.2)

plt.suptitle('Peterson-McCabe: Sentence Contrast in Free Personal Narratives',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig1_petmcc_contrast.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nAge correlations:")
for w in WINDOWS:
    col = f'contrast_W{w}'
    if col in df.columns:
        valid = df[[col, 'age_months']].dropna()
        r, p = stats.pearsonr(valid['age_months'], valid[col])
        sig = '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else ''
        print(f"  W{w:<4}: r={r:+.3f}, p={p:.4f} {sig}")